In [1]:
# ============================================================
# Hotel Reviews Sentiment Analysis
# Notebook 1: Data Preprocessing
# Dataset: La Veranda Hotel - Booking.com Reviews
# ============================================================

# ============================================================
# SECTION 1: Install Libraries
# ============================================================

!pip install nltk
!pip install textblob
!pip install vaderSentiment
!pip install wordcloud
!pip install seaborn
!pip install gensim
!pip install joblib







[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ============================================================
# SECTION 2: Import Libraries
# ============================================================

import re
import string
import logging
import warnings

import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from textblob import TextBlob
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')

In [3]:
# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /Users/paco/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/paco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/paco/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
# ============================================================
# SECTION 3: Load Dataset
# ============================================================

df = pd.read_csv('Dataset/La_Veranda_Reviews-2023-01-16.csv')
df.head()

,Title,PositiveReview,NegativeReview,Score,GuestName,GuestCountry,RoomType,NumberOfNights,VisitDate,GroupType,PropertyResponse
0,Wonderful place to stay.,"New, comfortable apartments, close to the airp...",Nothing at all.,10.0,Olga,Norway,Budget Twin Room,1 night,June 2022,Solo traveler,NaN
1,It was superb,We had a really pleasant stay! The staff was v...,NaN,10.0,Iwona,Poland,Double Room,3 nights,December 2022,Family,NaN
2,Very Good,the location is great and near the airport. bu...,NaN,8.0,Ruijia,Sweden,Double Room,1 night,December 2022,Solo traveler,NaN
3,Wonderful,Great stuff\nGreat Quality/price\nClean,NaN,9.0,Theprincem,United Kingdom,Double Room with Balcony,2 nights,September 2022,Solo traveler,NaN
4,"Fantastic value for a new, modern and spotless...","Clean and modern with very comfortable beds, i...",NaN,10.0,M,Switzerland,Family Suite with Balcony,1 night,October 2022,Family,NaN


In [5]:
# Check exact column names
print(df.columns.tolist())

['Title', 'PositiveReview', 'NegativeReview', 'Score', 'GuestName', 'GuestCountry', 'RoomType', 'NumberOfNights', 'VisitDate', 'GroupType', 'PropertyResponse']


In [6]:
# ============================================================
# SECTION 4: Initial Exploration
# ============================================================

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nNull Values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

Shape: (1523, 11)

Columns: ['Title', 'PositiveReview', 'NegativeReview', 'Score', 'GuestName', 'GuestCountry', 'RoomType', 'NumberOfNights', 'VisitDate', 'GroupType', 'PropertyResponse']

Data Types:
 Title                   str
PositiveReview          str
NegativeReview          str
Score               float64
GuestName               str
GuestCountry            str
RoomType                str
NumberOfNights          str
VisitDate               str
GroupType               str
PropertyResponse        str
dtype: object

Null Values:
 Title                  2
PositiveReview       748
NegativeReview      1089
Score                  0
GuestName              0
GuestCountry           0
RoomType              63
NumberOfNights         0
VisitDate              0
GroupType              0
PropertyResponse    1400
dtype: int64

Duplicates: 0


In [7]:
df.describe()

,Score
count,1523.000000
mean,8.973802
std,1.300178
min,1.000000
25%,8.000000
50%,9.000000
75%,10.000000
max,10.000000


In [8]:
# ============================================================
# SECTION 5: Drop Unnecessary Columns
# Keep only what's needed for sentiment analysis
# ============================================================

# We keep: Positive Review, Negative Review, Score, 
#          Guest Country, Room Type, Number of Nights,
#          Visit Date, Group Type
# We drop: Guest Name (not analytically useful), 
#          Property Response (hotel's voice, not guest's)

df = df.drop(columns=['Guest Name', 'Property Response'], errors='ignore')
df.head()

,Title,PositiveReview,NegativeReview,Score,GuestName,GuestCountry,RoomType,NumberOfNights,VisitDate,GroupType,PropertyResponse
0,Wonderful place to stay.,"New, comfortable apartments, close to the airp...",Nothing at all.,10.0,Olga,Norway,Budget Twin Room,1 night,June 2022,Solo traveler,NaN
1,It was superb,We had a really pleasant stay! The staff was v...,NaN,10.0,Iwona,Poland,Double Room,3 nights,December 2022,Family,NaN
2,Very Good,the location is great and near the airport. bu...,NaN,8.0,Ruijia,Sweden,Double Room,1 night,December 2022,Solo traveler,NaN
3,Wonderful,Great stuff\nGreat Quality/price\nClean,NaN,9.0,Theprincem,United Kingdom,Double Room with Balcony,2 nights,September 2022,Solo traveler,NaN
4,"Fantastic value for a new, modern and spotless...","Clean and modern with very comfortable beds, i...",NaN,10.0,M,Switzerland,Family Suite with Balcony,1 night,October 2022,Family,NaN


In [9]:
# ============================================================
# SECTION 6: Handle Missing Values
# ============================================================

# Some guests only leave a positive OR negative review — that's fine
# We fill empty reviews with empty string so cleaning doesn't break
df['PositiveReview'] = df['PositiveReview'].fillna('').astype(str)
df['NegativeReview'] = df['NegativeReview'].fillna('').astype(str)

mask = (df['PositiveReview'].str.strip() == '') & \
       (df['NegativeReview'].str.strip() == '')
df = df[~mask]

print("Remaining rows after dropping empty reviews:", len(df))
print("\nNull values after cleaning:\n", df.isnull().sum())

Remaining rows after dropping empty reviews: 786

Null values after cleaning:
 Title                 0
PositiveReview        0
NegativeReview        0
Score                 0
GuestName             0
GuestCountry          0
RoomType             27
NumberOfNights        0
VisitDate             0
GroupType             0
PropertyResponse    665
dtype: int64


In [10]:
# ============================================================
# SECTION 7: Text Cleaning Functions
# ============================================================

stop_words = set(stopwords.words('english'))

def remove_urls(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def remove_mentions_hashtags(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return re.sub(r'\@\w+|\#\w+', '', text)

def remove_punctuations(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_stopwords(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return ' '.join([w for w in text.split() if w not in stop_words])

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F700-\U0001F77F"
        u"\U0001F780-\U0001F7FF"
        u"\U0001F800-\U0001F8FF"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA00-\U0001FA6F"
        u"\U0001FA70-\U0001FAFF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def handle_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def lemmatize(text):
    lemmatizer = nltk.stem.WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

def stem(text):
    stemmer = nltk.stem.PorterStemmer()
    return ' '.join([stemmer.stem(word) for word in text.split()])

def clean_text(text):
    text = str(text).lower()
    text = remove_urls(text)
    text = remove_mentions_hashtags(text)
    text = remove_punctuations(text)
    text = remove_stopwords(text)
    text = remove_numbers(text)
    text = remove_emojis(text)
    text = handle_whitespace(text)
    text = lemmatize(text)
    text = stem(text)
    return text

In [11]:
# ============================================================
# SECTION 8: Apply Cleaning to Both Review Columns Separately
# This is the key difference from the original AirBnb notebook —
# we treat positive and negative reviews as independent text streams
# ============================================================

df['cleaned_positive'] = df['PositiveReview'].apply(clean_text)
df['cleaned_negative'] = df['NegativeReview'].apply(clean_text)

df[['PositiveReview', 'cleaned_positive', 
    'NegativeReview', 'cleaned_negative']].head()

,PositiveReview,cleaned_positive,NegativeReview,cleaned_negative
0,"New, comfortable apartments, close to the airp...",new comfort apart close airport clean beach st...,Nothing at all.,noth
1,We had a really pleasant stay! The staff was v...,realli pleasant stay staff nice help room clea...,,
2,the location is great and near the airport. bu...,locat great near airport bu stop close,,
3,Great stuff\nGreat Quality/price\nClean,great stuff great qualitypric clean,,
4,"Clean and modern with very comfortable beds, i...",clean modern comfort bed conveni locat easi st...,,


In [12]:
# ============================================================
# SECTION 9: Score-Based Sentiment Labeling
# 
# Unlike the original which inferred sentiment purely from text,
# we use the Score column (1-10) as ground truth:
#   Score 1-4  → negative
#   Score 5-7  → neutral
#   Score 8-10 → positive
#
# We apply this separately for positive and negative review tracks
# ============================================================

def score_to_sentiment(score):
    if score <= 4:
        return 'negative'
    elif score <= 7:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(score_to_sentiment)
print(df['sentiment'].value_counts())

sentiment
positive    714
neutral      63
negative      9
Name: count, dtype: int64


In [13]:
# ============================================================
# SECTION 10: Also Compute TextBlob Polarity
# (for comparison/validation against Score-based labels)
# ============================================================

# Polarity on positive reviews
df['polarity_positive'] = df['cleaned_positive'].apply(
    lambda x: TextBlob(x).sentiment.polarity if x.strip() != '' else 0.0
)

# Polarity on negative reviews
df['polarity_negative'] = df['cleaned_negative'].apply(
    lambda x: TextBlob(x).sentiment.polarity if x.strip() != '' else 0.0
)

df[['Score', 'sentiment', 
    'polarity_positive', 'polarity_negative']].head(10)

,Score,sentiment,polarity_positive,polarity_negative
0,10.0,positive,0.263258,0.0
1,10.0,positive,0.400000,0.0
2,8.0,positive,0.450000,0.0
3,9.0,positive,0.655556,0.0
4,10.0,positive,0.491667,0.0
5,9.0,positive,0.366667,0.0
6,8.0,positive,0.800000,0.0
7,10.0,positive,0.550000,0.0
8,10.0,positive,-0.125000,0.3
9,10.0,positive,0.333333,0.0


In [14]:
# ============================================================
# SECTION 11: Fix Date Column
# ============================================================

df = df[df['VisitDate'].str.contains('^\w', na=False)]
df['VisitDate'] = pd.to_datetime(df['VisitDate'], errors='coerce')
df = df.dropna(subset=['VisitDate'])

df['visit_month'] = df['VisitDate'].dt.month
df['visit_year'] = df['VisitDate'].dt.year

print("Date range:", df['VisitDate'].min(), "→", df['VisitDate'].max())

Date range: 2021-05-01 00:00:00 → 2022-12-01 00:00:00


In [15]:
# ============================================================
# SECTION 12: Remove Duplicates
# ============================================================

print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())

Duplicates before: 0
Duplicates after: 0


In [16]:
# ============================================================
# SECTION 13: Final Dataset Overview
# ============================================================

print("Final shape:", df.shape)
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())
print("\nColumns:", df.columns.tolist())
df.head()

Final shape: (786, 18)

Sentiment distribution:
sentiment
positive    714
neutral      63
negative      9
Name: count, dtype: int64

Columns: ['Title', 'PositiveReview', 'NegativeReview', 'Score', 'GuestName', 'GuestCountry', 'RoomType', 'NumberOfNights', 'VisitDate', 'GroupType', 'PropertyResponse', 'cleaned_positive', 'cleaned_negative', 'sentiment', 'polarity_positive', 'polarity_negative', 'visit_month', 'visit_year']


,Title,PositiveReview,NegativeReview,Score,GuestName,GuestCountry,RoomType,NumberOfNights,VisitDate,GroupType,PropertyResponse,cleaned_positive,cleaned_negative,sentiment,polarity_positive,polarity_negative,visit_month,visit_year
0,Wonderful place to stay.,"New, comfortable apartments, close to the airp...",Nothing at all.,10.0,Olga,Norway,Budget Twin Room,1 night,2022-06-01,Solo traveler,NaN,new comfort apart close airport clean beach st...,noth,positive,0.263258,0.0,6,2022
1,It was superb,We had a really pleasant stay! The staff was v...,,10.0,Iwona,Poland,Double Room,3 nights,2022-12-01,Family,NaN,realli pleasant stay staff nice help room clea...,,positive,0.400000,0.0,12,2022
2,Very Good,the location is great and near the airport. bu...,,8.0,Ruijia,Sweden,Double Room,1 night,2022-12-01,Solo traveler,NaN,locat great near airport bu stop close,,positive,0.450000,0.0,12,2022
3,Wonderful,Great stuff\nGreat Quality/price\nClean,,9.0,Theprincem,United Kingdom,Double Room with Balcony,2 nights,2022-09-01,Solo traveler,NaN,great stuff great qualitypric clean,,positive,0.655556,0.0,9,2022
4,"Fantastic value for a new, modern and spotless...","Clean and modern with very comfortable beds, i...",,10.0,M,Switzerland,Family Suite with Balcony,1 night,2022-10-01,Family,NaN,clean modern comfort bed conveni locat easi st...,,positive,0.491667,0.0,10,2022


In [17]:
# ============================================================
# SECTION 14: Save Cleaned Data
# Two files — one for positive review track, one for negative
# ============================================================

# Full cleaned dataset (used by ML/DL/LLM notebooks)
df.to_csv('Dataset/cleaned/hotel_reviews_cleaned.csv', index=False)
# Positive review track (for focused analysis)
pos_df = df[['cleaned_positive', 'polarity_positive', 
             'sentiment', 'Score', 'GuestCountry', 
             'RoomType', 'GroupType', 'visit_month', 'visit_year']]
pos_df.to_csv('Dataset/cleaned/positive_reviews_cleaned.csv', index=False)

# Negative review track
neg_df = df[['cleaned_negative', 'polarity_negative', 
             'sentiment', 'Score', 'GuestCountry', 
             'RoomType', 'GroupType', 'visit_month', 'visit_year']]
neg_df.to_csv('Dataset/cleaned/negative_reviews_cleaned.csv', index=False)

print("All files saved successfully!")

All files saved successfully!
